In [0]:
# ---------------------------------------------------------------------------
# LINEAGE METADATA — end-to-end provenance record for every table in the
# pipeline: source, retrieval, row counts at each stage, and known issues
# caught along the way. This is the "where did it come from, what happened
# to it, where did it go" traceability layer.
# ---------------------------------------------------------------------------
lineage_rows = [
    # (table_name, pipeline_stage, source_description, source_url, retrieval_date,
    #  row_count, upstream_table, known_issues, validation_result)
    ("bronze_brfss_2024_raw", "Bronze",
     "2024 BRFSS Data (ASCII), fixed-width, downloaded manually and uploaded to Unity Catalog Volume",
     "https://www.cdc.gov/brfss/annual_data/annual_2024.html", "2026-08-15",
     457670, "N/A (raw source)",
     "_AIDTST4 variable absent from file vs. CDC-documented layout (file is 2061 chars, documented layout implies 2062+) -- "
     "likely connected to the executive-order-driven data modification disclosed on CDC's BRFSS page",
     "Row count matches CDC's stated 457,670 exactly; Tennessee correctly absent per CDC's documented exclusion"),

    ("silver_brfss_2024_clean", "Silver",
     "Cleaned/typed/imputed BRFSS Bronze data", "N/A (derived)", "2026-08-15",
     416081, "bronze_brfss_2024_raw",
     "None beyond upstream Bronze issue",
     "Unweighted national inactivity rate 22.51% vs. manuscript's 22.5% (0.01pt gap); INCOME3 null count "
     "matched manuscript's cited 87,423 exactly"),

    ("gold_brfss_2024_harmonized", "Gold",
     "BRFSS mapped into common cross-system schema", "N/A (derived)", "2026-08-16",
     416081, "silver_brfss_2024_clean",
     "_RACE value-label mapping NOT independently verified against 2024 codebook (used standard historical convention)",
     "Race-by-prevalence pattern (Hispanic highest, 29.0%) directionally matches manuscript's reported 30.1%"),

    ("bronze_nyts_2025_raw", "Bronze",
     "2025 NYTS Dataset (SAS7BDAT), downloaded manually and uploaded to Unity Catalog Volume",
     "https://www.fda.gov/media/191376/download?attachment", "2026-08-16",
     23630, "N/A (raw source)",
     "None",
     "Weighted CELCIGT rate 5.23% vs. FDA's published 5.2%"),

    ("silver_nyts_2025_clean", "Silver",
     "Cleaned/typed NYTS Bronze data, letter-code missingness recoded", "N/A (derived)", "2026-08-16",
     23380, "bronze_nyts_2025_raw",
     "CAUGHT AND FIXED: initial cleaning pass used astype(str) during Bronze write, which converted real "
     "nulls into the literal string 'nan', silently bypassing missing-value recoding for 250 rows "
     "(first attempt gave 5.18% instead of correct 5.23%). Fixed by explicitly treating 'nan' as a missing code.",
     "Weighted CELCIGT rate 5.23% post-fix, matching Bronze-layer check and FDA's 5.2% published figure"),

    ("gold_nyts_2025_harmonized", "Gold",
     "NYTS mapped into common cross-system schema, race flags collapsed to single category", "N/A (derived)", "2026-08-16",
     23380, "silver_nyts_2025_clean",
     "No state-level geography available in NYTS public-use file (documented, not a parsing gap); "
     "small subgroup sizes for NHPI (n=66) and MENA (n=298) carry wide uncertainty if reported individually",
     "Prevalence-by-sex and by-race breakdowns computed via shared cross-system function, no source-specific code"),

    ("gold_schema_crosswalk_metadata", "Gold (metadata)",
     "Documents every variable mapping between BRFSS and NYTS harmonized schemas", "N/A (derived)", "2026-08-16",
     14, "N/A (metadata table, not row-level data)",
     "One entry (BRFSS _RACE) flagged as needing codebook verification",
     "N/A -- this table IS the validation record for the harmonization logic itself"),
]

lineage_schema = ["table_name", "pipeline_stage", "source_description", "source_url", "retrieval_date",
                   "row_count", "upstream_table", "known_issues", "validation_result"]

lineage_df = spark.createDataFrame(lineage_rows, schema=lineage_schema)

lineage_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_pipeline_lineage")
print(f"Written {lineage_df.count()} lineage entries")
lineage_df.select("table_name", "pipeline_stage", "row_count", "upstream_table").show(20, truncate=False)

Written 7 lineage entries
+------------------------------+---------------+---------+----------------------------------------+
|table_name                    |pipeline_stage |row_count|upstream_table                          |
+------------------------------+---------------+---------+----------------------------------------+
|bronze_brfss_2024_raw         |Bronze         |457670   |N/A (raw source)                        |
|silver_brfss_2024_clean       |Silver         |416081   |bronze_brfss_2024_raw                   |
|gold_brfss_2024_harmonized    |Gold           |416081   |silver_brfss_2024_clean                 |
|bronze_nyts_2025_raw          |Bronze         |23630    |N/A (raw source)                        |
|silver_nyts_2025_clean        |Silver         |23380    |bronze_nyts_2025_raw                    |
|gold_nyts_2025_harmonized     |Gold           |23380    |silver_nyts_2025_clean                  |
|gold_schema_crosswalk_metadata|Gold (metadata)|14       |N/A (metadata ta

In [0]:
# The full traceability query: trace any table back to its raw source
# in one call, no need to read notebooks to reconstruct history
lineage_df.select("table_name", "pipeline_stage", "source_url", "known_issues").show(20, truncate=80)

+------------------------------+---------------+------------------------------------------------------+--------------------------------------------------------------------------------+
|                    table_name| pipeline_stage|                                            source_url|                                                                    known_issues|
+------------------------------+---------------+------------------------------------------------------+--------------------------------------------------------------------------------+
|         bronze_brfss_2024_raw|         Bronze|https://www.cdc.gov/brfss/annual_data/annual_2024.html|_AIDTST4 variable absent from file vs. CDC-documented layout (file is 2061 ch...|
|       silver_brfss_2024_clean|         Silver|                                         N/A (derived)|                                               None beyond upstream Bronze issue|
|    gold_brfss_2024_harmonized|           Gold|                           